In [44]:
# importamos librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F # para seleccionar filas
from pyspark.sql.types import StructType, IntegerType, StringType, StructField, DoubleType

In [45]:
# creamos sesion
spark = (SparkSession.builder 
            .appName("aplicación_1")  #nombre aplicacion
            .master("local[*]")
            .config("spark.driver.memory", "2g")
            .getOrCreate()
         )
print(f"version spark: {spark.version}")

version spark: 4.2.0


In [46]:
# lectura dataset (NO USAR EN PRODUCCIÓN)
df = spark.read.csv("02.sales.csv", header=True, inferSchema=True, sep=",")
df.show(5, truncate=False)

+---+----------+--------+------+--------+
|id |date      |product |price |quantity|
+---+----------+--------+------+--------+
|1  |2024-01-10|Portátil|1200.0|1       |
|2  |2024-01-11|Ratón   |25.0  |3       |
|3  |2024-01-12|Teclado |45.0  |2       |
|4  |2024-01-13|Monitor |300.0 |1       |
|5  |2024-01-14|Portátil|1250.0|1       |
+---+----------+--------+------+--------+
only showing top 5 rows


In [47]:
# Mostramos esquema
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



In [48]:
# Para sistemas en producción --> definimos schema de forma manual
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

# lectura + aplicamos schema
df_produccion = spark.read.csv("02.sales.csv", header=True, schema= schema, sep=",")

df_produccion.show(5, truncate= False)

+---+----------+--------+------+--------+
|id |date      |product |price |quantity|
+---+----------+--------+------+--------+
|1  |2024-01-10|Portátil|1200.0|1       |
|2  |2024-01-11|Ratón   |25.0  |3       |
|3  |2024-01-12|Teclado |45.0  |2       |
|4  |2024-01-13|Monitor |300.0 |1       |
|5  |2024-01-14|Portátil|1250.0|1       |
+---+----------+--------+------+--------+
only showing top 5 rows


In [49]:
# cambiamos tipo fecha
# funcion withColumn se usa para trabajar sobre una columna, argumentos(columna_nueva/vieja, operacion)
df_produccion = df_produccion.withColumn("date", F.to_date("date", "yyyy-mm-dd")) 
df_produccion.show(5, truncate= False)

+---+----------+--------+------+--------+
|id |date      |product |price |quantity|
+---+----------+--------+------+--------+
|1  |2024-01-10|Portátil|1200.0|1       |
|2  |2024-01-11|Ratón   |25.0  |3       |
|3  |2024-01-12|Teclado |45.0  |2       |
|4  |2024-01-13|Monitor |300.0 |1       |
|5  |2024-01-14|Portátil|1250.0|1       |
+---+----------+--------+------+--------+
only showing top 5 rows


In [50]:
df_produccion.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



In [51]:
# transformación select: seleccionar n variables del conjunto total
df_select = df_produccion.select("id", "date", "price", "quantity")
df_select.show(3)

+---+----------+------+--------+
| id|      date| price|quantity|
+---+----------+------+--------+
|  1|2024-01-10|1200.0|       1|
|  2|2024-01-11|  25.0|       3|
|  3|2024-01-12|  45.0|       2|
+---+----------+------+--------+
only showing top 3 rows


In [52]:
# creamos una nueva columna siendo resultado de otras
df_select = df_select.withColumn("total", F.col("price") * F.col("quantity"))
df_select.show(4)

+---+----------+------+--------+------+
| id|      date| price|quantity| total|
+---+----------+------+--------+------+
|  1|2024-01-10|1200.0|       1|1200.0|
|  2|2024-01-11|  25.0|       3|  75.0|
|  3|2024-01-12|  45.0|       2|  90.0|
|  4|2024-01-13| 300.0|       1| 300.0|
+---+----------+------+--------+------+
only showing top 4 rows


In [53]:
# filtrar filas --> filter
df_filtado = df_select.filter(F.col("total") > 100.00)
df_filtado.show(5)

+---+----------+------+--------+------+
| id|      date| price|quantity| total|
+---+----------+------+--------+------+
|  1|2024-01-10|1200.0|       1|1200.0|
|  4|2024-01-13| 300.0|       1| 300.0|
|  5|2024-01-14|1250.0|       1|1250.0|
|  7|2024-01-16| 290.0|       2| 580.0|
|  9|2024-01-18|  80.0|       5| 400.0|
+---+----------+------+--------+------+
only showing top 5 rows


In [54]:
# Cerramos sesión spark
spark.stop()